# 02 - DataLoader para Secuencias de Video (WLASL)

## Objetivo
Crear una clase `WLASLDataset` que permita cargar secuencias de frames como tensores para entrenar modelos de reconocimiento de palabras en lengua de signos.

## ¿Por qué necesitamos esto?

PyTorch necesita que los datos estén en un formato específico para entrenar:
- Los datos deben venir en **batches** (grupos de ejemplos)
- Cada ejemplo debe ser un **tensor** (array multidimensional)
- Todos los ejemplos de un batch deben tener **la misma forma**

### El problema con videos
Nuestros videos tienen **diferente número de frames**:
- Video A: 30 frames
- Video B: 45 frames  
- Video C: 60 frames

¿Cómo metemos esto en un batch? No podemos simplemente apilarlos.

### La solución: Padding
Rellenamos los videos cortos con "padding" (ceros) hasta que todos tengan la misma longitud:
```
Video A: [f1, f2, ..., f30, PAD, PAD, ..., PAD]  → 60 frames
Video B: [f1, f2, ..., f45, PAD, PAD, ..., PAD]  → 60 frames
Video C: [f1, f2, ..., f60]                      → 60 frames
```

Además, guardamos las **longitudes originales** para que el LSTM sepa cuándo ignorar el padding.

## 0. Configuración del Entorno (Colab vs Local)

Esta celda detecta automáticamente el entorno y configura las rutas.

### Opción A: Colab Web (Recomendado para entrenar)
1. Abre https://colab.research.google.com
2. Sube este notebook o ábrelo desde Drive
3. Sube la carpeta `wlasl30` a: `Mi unidad/tfg/datasets/wlasl30/`
4. Ejecuta - Drive se montará automáticamente

### Opción B: VSCode con extensión Colab
La extensión VSCode de Colab **NO soporta drive.mount()** ([issue conocido](https://github.com/anthropics/courses/issues/76)).

Para usar esta opción:
1. Ve a tu carpeta `wlasl30` en Google Drive
2. Clic derecho → "Compartir" → "Cualquier persona con el enlace puede ver"
3. Copia el ID de la URL: `https://drive.google.com/drive/folders/XXXXX` ← ese `XXXXX` es el ID
4. Pégalo en `DRIVE_FOLDER_ID` en la celda de abajo
5. El notebook descargará el dataset automáticamente con `gdown`

### Opción C: Local (tu máquina)
El dataset debe estar en: `06_datasets/wlasl/processed/wlasl30/`

In [ ]:
import os
import subprocess
from pathlib import Path

# ============================================
# CONFIGURACIÓN PARA COLAB VSCODE EXTENSION
# ============================================
# Si usas la extensión VSCode de Colab, drive.mount() NO funciona.
# Opción 1: Usar Colab web normal (recomendado para entrenar)
# Opción 2: Descargar dataset con gdown (configurar abajo)

# Si quieres usar gdown, comparte tu carpeta wlasl30 en Drive y pon el ID aquí:
# (El ID está en la URL: https://drive.google.com/drive/folders/XXXXX ← ese es el ID)
DRIVE_FOLDER_ID = ""  # <-- PON AQUÍ TU ID si usas VSCode extension

# ============================================
# DETECTAR ENTORNO: COLAB vs LOCAL
# ============================================

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    print("🚀 Ejecutando en Google Colab")
    
    # Intentar montar Drive (funciona en Colab web, NO en VSCode extension)
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_ms=15000)
        USE_DRIVE_MOUNT = True
        print("✅ Drive montado correctamente")
    except Exception as e:
        USE_DRIVE_MOUNT = False
        print(f"⚠️ No se pudo montar Drive: {e}")
        print("   (Esto es normal si usas la extensión VSCode de Colab)")
    
    if USE_DRIVE_MOUNT:
        # ============================================
        # OPCIÓN A: COLAB WEB CON DRIVE MONTADO
        # ============================================
        DRIVE_BASE = Path("/content/drive/MyDrive/tfg")
        DATASET_ROOT = DRIVE_BASE / "datasets" / "wlasl30"
        EXPERIMENTS_DIR = DRIVE_BASE / "experiments" / "word_recognition"
        print(f"\n📁 Usando Drive montado:")
        
    else:
        # ============================================
        # OPCIÓN B: COLAB VSCODE EXTENSION CON GDOWN
        # ============================================
        print("\n📥 Intentando descargar dataset con gdown...")
        
        if not DRIVE_FOLDER_ID:
            print("\n❌ ERROR: Para usar VSCode extension necesitas configurar DRIVE_FOLDER_ID")
            print("\nPasos:")
            print("  1. Ve a Google Drive y encuentra tu carpeta wlasl30")
            print("  2. Clic derecho → Compartir → Cualquier persona con el enlace")
            print("  3. Copia el ID de la URL (la parte después de /folders/)")
            print("  4. Pégalo arriba en DRIVE_FOLDER_ID = 'tu_id_aqui'")
            print("\n💡 ALTERNATIVA RECOMENDADA: Usa Colab web normal para entrenar")
            print("   Abre: https://colab.research.google.com y sube este notebook")
            raise ValueError("DRIVE_FOLDER_ID no configurado")
        
        # Instalar gdown si no está
        subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
        import gdown
        
        # Descargar carpeta de Drive
        DATASET_ROOT = Path("/content/wlasl30")
        EXPERIMENTS_DIR = Path("/content/experiments/word_recognition")
        
        if not DATASET_ROOT.exists():
            print(f"⬇️ Descargando dataset desde Drive...")
            gdown.download_folder(
                id=DRIVE_FOLDER_ID,
                output=str(DATASET_ROOT),
                quiet=False
            )
            print("✅ Dataset descargado")
        else:
            print("✅ Dataset ya existe en /content/wlasl30")

else:
    print("💻 Ejecutando en entorno local")
    
    # ============================================
    # RUTAS LOCALES
    # ============================================
    DATASET_ROOT = Path("../../06_datasets/wlasl/processed/wlasl30")
    EXPERIMENTS_DIR = Path("../../05_experiments/word_recognition")

# Crear directorio de experimentos
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

# Verificar que existe el dataset
print(f"\n📂 Rutas configuradas:")
print(f"   Dataset: {DATASET_ROOT}")
print(f"   Experimentos: {EXPERIMENTS_DIR}")

if not DATASET_ROOT.exists():
    print(f"\n❌ ERROR: No se encontró el dataset en {DATASET_ROOT}")
    if IN_COLAB and USE_DRIVE_MOUNT:
        print("\n¿Has subido el dataset a Google Drive?")
        print("Estructura esperada en Drive:")
        print("  Mi unidad/")
        print("  └── tfg/")
        print("      └── datasets/")
        print("          └── wlasl30/")
        print("              ├── metadata.json")
        print("              ├── train/")
        print("              ├── val/")
        print("              └── test/")
    raise FileNotFoundError(f"Dataset no encontrado: {DATASET_ROOT}")
else:
    print(f"\n✅ Dataset encontrado correctamente")

## 1. Imports y Configuración

In [ ]:
import json
from typing import List, Tuple, Dict, Optional

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchvision.transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Verificar GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Cargar Metadata del Dataset

El archivo `metadata.json` contiene información sobre el dataset:
- Lista de palabras (glosses)
- Mapeo palabra → índice numérico
- Estadísticas (número de videos, frames, etc.)

In [ ]:
# Cargar metadata
with open(DATASET_ROOT / "metadata.json", "r") as f:
    metadata = json.load(f)

print("=== Información del Dataset ===")
print(f"Nombre: {metadata['name']}")
print(f"Total de palabras: {metadata['statistics']['total_words']}")
print(f"Total de secuencias: {metadata['statistics']['total_sequences']}")
print(f"  - Train: {metadata['statistics']['train']}")
print(f"  - Val: {metadata['statistics']['val']}")
print(f"  - Test: {metadata['statistics']['test']}")
print(f"\nFrames por video:")
print(f"  - Mínimo: {metadata['statistics']['frames_min']}")
print(f"  - Máximo: {metadata['statistics']['frames_max']}")
print(f"  - Media: {metadata['statistics']['frames_mean']:.1f}")
print(f"\nPalabras: {metadata['glosses']}")

## 3. La Clase WLASLDataset

### ¿Qué es un Dataset en PyTorch?

Un `Dataset` es una clase que implementa dos métodos:
- `__len__()`: Devuelve cuántos ejemplos hay
- `__getitem__(idx)`: Devuelve el ejemplo en la posición `idx`

PyTorch usa estos métodos para iterar sobre los datos durante el entrenamiento.

### ¿Qué debe devolver nuestro Dataset?

Para cada video, devolvemos:
1. **frames**: Tensor de forma `[T, C, H, W]` donde:
   - T = número de frames
   - C = canales (3 para RGB)
   - H, W = alto y ancho (224x224)
2. **label**: Índice numérico de la palabra (0-29 para WLASL30)
3. **length**: Número real de frames (sin padding)

In [ ]:
class WLASLDataset(Dataset):
    """
    Dataset para cargar secuencias de video del dataset WLASL.
    
    Cada video es una carpeta con frames (0000.jpg, 0001.jpg, ...).
    El Dataset carga todos los frames de un video y los devuelve como tensor.
    
    Args:
        root_dir: Ruta al dataset (ej: wlasl30/)
        split: 'train', 'val', o 'test'
        transform: Transformaciones a aplicar a cada frame
        max_frames: Máximo de frames a cargar (None = todos)
        sample_strategy: 'all' (todos los frames) o 'uniform' (muestreo uniforme)
    """
    
    def __init__(
        self,
        root_dir: str,
        split: str = "train",
        transform: Optional[T.Compose] = None,
        max_frames: Optional[int] = None,
        sample_strategy: str = "all"
    ):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform
        self.max_frames = max_frames
        self.sample_strategy = sample_strategy
        
        # Cargar metadata
        with open(self.root_dir / "metadata.json", "r") as f:
            self.metadata = json.load(f)
        
        # Mapeo palabra → índice
        self.gloss_to_idx = self.metadata["gloss_to_idx"]
        self.idx_to_gloss = {v: k for k, v in self.gloss_to_idx.items()}
        self.num_classes = len(self.gloss_to_idx)
        
        # Construir lista de (video_path, label)
        self.samples = self._build_sample_list()
        
        print(f"WLASLDataset [{split}]: {len(self.samples)} videos, {self.num_classes} clases")
    
    def _build_sample_list(self) -> List[Tuple[Path, int]]:
        """
        Construye la lista de todos los videos con sus etiquetas.
        
        Estructura esperada:
        root_dir/
          train/
            book/
              07068/  ← carpeta de video
                0000.jpg
                0001.jpg
                ...
        """
        samples = []
        split_dir = self.root_dir / self.split
        
        if not split_dir.exists():
            raise ValueError(f"No existe el directorio {split_dir}")
        
        # Iterar por cada palabra (gloss)
        for gloss_dir in sorted(split_dir.iterdir()):
            if not gloss_dir.is_dir():
                continue
            
            gloss = gloss_dir.name
            if gloss not in self.gloss_to_idx:
                print(f"Advertencia: '{gloss}' no está en el vocabulario")
                continue
            
            label = self.gloss_to_idx[gloss]
            
            # Iterar por cada video de esta palabra
            for video_dir in sorted(gloss_dir.iterdir()):
                if video_dir.is_dir():
                    samples.append((video_dir, label))
        
        return samples
    
    def _load_frames(self, video_dir: Path) -> torch.Tensor:
        """
        Carga todos los frames de un video y los convierte a tensor.
        
        Returns:
            Tensor de forma [T, C, H, W]
        """
        # Obtener lista de frames ordenada
        frame_files = sorted(video_dir.glob("*.jpg"))
        
        if len(frame_files) == 0:
            raise ValueError(f"No hay frames en {video_dir}")
        
        # Aplicar estrategia de muestreo si hay max_frames
        if self.max_frames and len(frame_files) > self.max_frames:
            if self.sample_strategy == "uniform":
                # Muestreo uniforme: seleccionar frames equiespaciados
                indices = np.linspace(0, len(frame_files) - 1, self.max_frames, dtype=int)
                frame_files = [frame_files[i] for i in indices]
            else:
                # Por defecto: truncar (tomar los primeros)
                frame_files = frame_files[:self.max_frames]
        
        # Cargar cada frame
        frames = []
        for frame_path in frame_files:
            img = Image.open(frame_path).convert("RGB")
            
            if self.transform:
                img = self.transform(img)
            else:
                # Transformación por defecto: convertir a tensor
                img = T.ToTensor()(img)
            
            frames.append(img)
        
        # Apilar frames: [T, C, H, W]
        return torch.stack(frames)
    
    def __len__(self) -> int:
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, int]:
        """
        Returns:
            frames: Tensor [T, C, H, W] con los frames del video
            label: Índice de la clase (palabra)
            length: Número de frames (útil para padding)
        """
        video_dir, label = self.samples[idx]
        frames = self._load_frames(video_dir)
        length = frames.shape[0]  # Número de frames
        
        return frames, label, length
    
    def get_gloss(self, idx: int) -> str:
        """Convierte índice numérico a palabra."""
        return self.idx_to_gloss[idx]

## 4. Transformaciones (Data Augmentation)

### ¿Por qué transformaciones?

1. **Normalización**: Las redes pre-entrenadas (ResNet) esperan imágenes normalizadas con media y std específicos de ImageNet
2. **Data Augmentation**: Aumentar artificialmente la variedad de datos para evitar overfitting

### Transformaciones que usaremos:

**Para entrenamiento:**
- Resize a 224x224 (tamaño que espera ResNet)
- Random horizontal flip (volteo aleatorio)
- Color jitter (variaciones de brillo/contraste)
- Normalización ImageNet

**Para validación/test:**
- Solo resize y normalización (sin augmentation)

In [ ]:
# Valores de normalización de ImageNet
# Estos valores son estándar y vienen de cómo se entrenó ResNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_transforms(split: str, img_size: int = 224):
    """
    Devuelve las transformaciones apropiadas para cada split.
    
    Args:
        split: 'train', 'val', o 'test'
        img_size: Tamaño de imagen (224 para ResNet)
    """
    if split == "train":
        return T.Compose([
            T.Resize((img_size, img_size)),
            # Data augmentation (solo en train)
            T.RandomHorizontalFlip(p=0.5),  # 50% de probabilidad de voltear
            T.ColorJitter(
                brightness=0.2,  # ±20% brillo
                contrast=0.2,    # ±20% contraste
                saturation=0.2,  # ±20% saturación
            ),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        # Validación y test: sin augmentation
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

print("Transformaciones definidas ✓")

## 5. Collate Function para Padding

### ¿Qué es una collate function?

Cuando el `DataLoader` crea un batch, necesita combinar múltiples ejemplos en tensores.
Por defecto, PyTorch intenta apilar todo con `torch.stack()`, pero esto falla si los tensores tienen tamaños diferentes.

Una **collate function** personalizada nos permite definir cómo combinar los ejemplos.

### Nuestra estrategia:

1. Encontrar la longitud máxima en el batch
2. Rellenar (pad) todas las secuencias hasta esa longitud
3. Guardar las longitudes originales para el LSTM

```
Antes:                          Después (con padding):
Video 1: [30, 3, 224, 224]  →   [60, 3, 224, 224] + length=30
Video 2: [45, 3, 224, 224]  →   [60, 3, 224, 224] + length=45
Video 3: [60, 3, 224, 224]  →   [60, 3, 224, 224] + length=60
```

In [ ]:
def collate_fn(batch: List[Tuple[torch.Tensor, int, int]]):
    """
    Función de colación que maneja secuencias de longitud variable.
    
    Args:
        batch: Lista de tuplas (frames, label, length)
    
    Returns:
        frames_padded: Tensor [B, T_max, C, H, W] con padding
        labels: Tensor [B] con las etiquetas
        lengths: Tensor [B] con las longitudes originales
    """
    # Separar los componentes
    frames_list, labels, lengths = zip(*batch)
    
    # Convertir a tensores
    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    
    # Encontrar la longitud máxima en este batch
    max_len = max(lengths).item()
    
    # Padding manual para secuencias de video
    # pad_sequence de PyTorch funciona para [T, *], pero aquí tenemos [T, C, H, W]
    batch_size = len(frames_list)
    C, H, W = frames_list[0].shape[1:]  # Canales, alto, ancho
    
    # Crear tensor de salida con ceros (padding)
    frames_padded = torch.zeros(batch_size, max_len, C, H, W)
    
    # Copiar cada secuencia al tensor de salida
    for i, frames in enumerate(frames_list):
        seq_len = frames.shape[0]
        frames_padded[i, :seq_len] = frames
    
    return frames_padded, labels, lengths

print("Collate function definida ✓")

## 6. Crear los DataLoaders

### ¿Qué es un DataLoader?

El `DataLoader` es el que realmente itera sobre el `Dataset`:
- Crea batches de tamaño `batch_size`
- Mezcla los datos si `shuffle=True` (importante para train)
- Carga datos en paralelo con `num_workers`
- Aplica nuestra `collate_fn` para manejar el padding

In [ ]:
# Hiperparámetros
BATCH_SIZE = 4  # Pequeño porque los videos ocupan mucha memoria
NUM_WORKERS = 2  # Procesos paralelos para cargar datos (usar 0 en Colab si hay problemas)
MAX_FRAMES = 64  # Limitar frames para no quedarnos sin memoria

# En Colab a veces hay problemas con num_workers > 0
if IN_COLAB:
    NUM_WORKERS = 2  # Puedes bajar a 0 si hay errores

# Crear datasets
train_dataset = WLASLDataset(
    root_dir=DATASET_ROOT,
    split="train",
    transform=get_transforms("train"),
    max_frames=MAX_FRAMES,
    sample_strategy="uniform"  # Muestreo uniforme si hay más de MAX_FRAMES
)

val_dataset = WLASLDataset(
    root_dir=DATASET_ROOT,
    split="val",
    transform=get_transforms("val"),
    max_frames=MAX_FRAMES,
    sample_strategy="uniform"
)

test_dataset = WLASLDataset(
    root_dir=DATASET_ROOT,
    split="test",
    transform=get_transforms("test"),
    max_frames=MAX_FRAMES,
    sample_strategy="uniform"
)

In [ ]:
# Crear DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,  # Mezclar datos en cada época
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True  # Optimización para GPU
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # No mezclar en validación
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

print(f"\nDataLoaders creados:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val: {len(val_loader)} batches")
print(f"  Test: {len(test_loader)} batches")

## 7. Verificar que Todo Funciona

Vamos a cargar un batch y verificar las formas de los tensores.

In [ ]:
# Obtener un batch de ejemplo
frames, labels, lengths = next(iter(train_loader))

print("=== Verificación de un Batch ===")
print(f"Frames shape: {frames.shape}")
print(f"  → [batch_size, max_seq_len, channels, height, width]")
print(f"  → [{frames.shape[0]}, {frames.shape[1]}, {frames.shape[2]}, {frames.shape[3]}, {frames.shape[4]}]")
print(f"\nLabels shape: {labels.shape}")
print(f"  → Valores: {labels.tolist()}")
print(f"  → Palabras: {[train_dataset.get_gloss(l.item()) for l in labels]}")
print(f"\nLengths shape: {lengths.shape}")
print(f"  → Longitudes reales: {lengths.tolist()}")

In [ ]:
# Visualizar algunos frames de un video del batch
def denormalize(tensor):
    """Deshace la normalización de ImageNet para visualizar."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return tensor * std + mean

# Tomar el primer video del batch
video_idx = 0
video_frames = frames[video_idx]  # [T, C, H, W]
video_length = lengths[video_idx].item()
video_label = train_dataset.get_gloss(labels[video_idx].item())

# Mostrar 8 frames equiespaciados
n_show = min(8, video_length)
indices = np.linspace(0, video_length - 1, n_show, dtype=int)

fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2))
fig.suptitle(f"Palabra: '{video_label}' ({video_length} frames)", fontsize=12)

for i, idx in enumerate(indices):
    frame = denormalize(video_frames[idx])
    frame = frame.permute(1, 2, 0).numpy()  # [H, W, C]
    frame = np.clip(frame, 0, 1)  # Asegurar rango [0, 1]
    
    axes[i].imshow(frame)
    axes[i].set_title(f"Frame {idx}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 8. Guardar Configuración

Guardamos la configuración para referencia.

In [ ]:
# Resumen de configuración
config = {
    "dataset_root": str(DATASET_ROOT),
    "num_classes": train_dataset.num_classes,
    "batch_size": BATCH_SIZE,
    "max_frames": MAX_FRAMES,
    "img_size": 224,
    "sample_strategy": "uniform",
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
}

print("=== Configuración del DataLoader ===")
for k, v in config.items():
    print(f"  {k}: {v}")

## 9. Resumen y Siguiente Paso

### Lo que hemos creado:

1. **`WLASLDataset`**: Clase que carga videos como secuencias de tensores
2. **`get_transforms()`**: Función que devuelve las transformaciones apropiadas
3. **`collate_fn()`**: Función que maneja el padding de secuencias variables
4. **DataLoaders**: train_loader, val_loader, test_loader listos para usar

### Formas de los tensores:

```
frames:  [batch_size, max_seq_len, 3, 224, 224]
labels:  [batch_size]
lengths: [batch_size]
```

### Siguiente notebook: `03_cnn_lstm_training.ipynb`

En el siguiente notebook implementaremos:
1. CNN encoder (ResNet18) para extraer features de cada frame
2. LSTM para procesar la secuencia temporal
3. Clasificador para predecir la palabra
4. Loop de entrenamiento
5. Métricas (Top-1, Top-5, confusion matrix)